# Gaussian-process modeling, SHAP interpretation, and independent validation

**Reproducible analysis supporting the JACS Au manuscript revisions**

This notebook combines the original **GP yield/selectivity and SHAP** analysis with the **external hold-out validation and uncertainty calibration** analysis. The scientific procedures, modeling choices, and diagnostic figures are preserved from their original notebooks.

### Analysis overview

1. **Constrained-data model development:** six-variable GP regressions for BHET yield and BHET selectivity using a common 80:20 train/test split and five-fold cross-validation.
2. **Model interpretation:** model-agnostic SHAP explanations of the internal-split fitted GP models.
3. **Independent hold-out validation:** refit the same GP specification on **all 203 constrained observations**, then evaluate accuracy, calibration, uncertainty, and extrapolation on **237 independent hold-out observations**.

**Important distinction:** the GP models used for internal testing and SHAP are *not* the fully refitted GP models used for external validation. The external dataset is never used to fit either model.

### Required inputs

- `constrained_203.xlsx` — worksheet `constrained` (203 observations)
- `holdout_237.xlsx` — worksheet `clean` (237 observations)

Place both workbooks alongside this notebook or in the repository's `data/` folder. Outputs are written to `gp_outputs/` and `gp_calibration_outputs/` under the working project directory.

**Execution:** Restart the kernel and choose **Run All**. SHAP KernelExplainer calculations may be computationally expensive. Figures and tabular summaries are written to disk.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, DotProduct
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_SPLIT = 101
MODEL_RANDOM_STATE = 42
TEST_SIZE = 0.20
SHAP_RANDOM_STATE = 42

# Resolve inputs for both a flat folder and a GitHub notebooks/data layout.
ROOT = Path.cwd()

def resolve_input(filename):
    candidates = [
        ROOT / filename,
        ROOT / "data" / filename,
        ROOT.parent / filename,
        ROOT.parent / "data" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Checked: " + ", ".join(map(str, candidates))
    )

DATA_PATH = resolve_input("constrained_203.xlsx")
OUTPUT_DIR = ROOT / "gp_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Data:", DATA_PATH.resolve())
print("Outputs:", OUTPUT_DIR.resolve())

## Part I — Constrained-data GP development and internal validation

### 1. Load and check the constrained dataset

In [ ]:
data = pd.read_excel(DATA_PATH, sheet_name="constrained")

FEATURES = ["S", "time", "enzyme", "solvent", "temperature", "ph"]
TARGETS = {
    "yield": "Mean_BHET",
    "selectivity": "Mean_Selectivity_BHET",
}

required = FEATURES + list(TARGETS.values())
missing = [column for column in required if column not in data.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")
if data[required].isna().any().any():
    raise ValueError("Missing values are present in modelling columns.")
if data.duplicated(FEATURES).any():
    raise ValueError("Duplicate six-variable conditions found in the seed dataset.")

print("Dataset shape:", data.shape)
display(data[required].describe().T)

### Feature naming note

The original notebook uses `S` as an input. In the supplied seed workbook, `S` contains 32–512 and corresponds to substrate mass per reaction volume; `substrate / disc mL-1` contains the 1–16 disc-loading levels used in the manuscript. This notebook intentionally preserves the original yield-model input list. The symbol should be reconciled before final publication.

## 2. Common 80:20 split

In [ ]:
X = data[FEATURES].copy()
indices = np.arange(len(data))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_SPLIT,
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

print(f"Training rows: {len(train_idx)}")
print(f"Test rows: {len(test_idx)}")
print("Shared indices are used for yield and selectivity.")

### 3. Shared Gaussian-process model specification and evaluation helpers

The same kernel and regressor settings are reused for both response variables and later for the independent hold-out evaluation. The GP specification is intentionally unchanged from the original notebooks.

In [ ]:
def make_gp():
    # Submitted yield-model specification, reused without changing its settings.
    kernel = Matern(
        nu=3,
        length_scale_bounds=(1e-4, 1e5),
    ) + DotProduct(
        sigma_0=1.0,
        sigma_0_bounds=(1e-4, 1e5),
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        alpha=0.001,
        normalize_y=True,
        random_state=MODEL_RANDOM_STATE,
        n_restarts_optimizer=0,
    )


def fit_and_evaluate(target_name, target_column):
    y = data[target_column].copy()
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model = make_gp()
    model.fit(X_train, y_train)

    pred_train, std_train = model.predict(X_train, return_std=True)
    pred_test, std_test = model.predict(X_test, return_std=True)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")

    metrics = {
        "target": target_name,
        "train_r2": r2_score(y_train, pred_train),
        "test_r2": r2_score(y_test, pred_test),
        "train_rmse": mean_squared_error(y_train, pred_train) ** 0.5,
        "test_rmse": mean_squared_error(y_test, pred_test) ** 0.5,
        "cv_r2_mean": cv_scores.mean(),
        "cv_r2_sd": cv_scores.std(ddof=1),
        "kernel_after_fit": str(model.kernel_),
    }

    predictions = pd.DataFrame({
        "source_row": test_idx,
        "target": target_name,
        "observed": y_test.to_numpy(),
        "posterior_mean": pred_test,
        "posterior_sd": std_test,
        "residual": y_test.to_numpy() - pred_test,
        "standardized_residual": (y_test.to_numpy() - pred_test) / std_test,
    })
    for feature in FEATURES:
        predictions[feature] = X_test[feature].to_numpy()

    return {
        "model": model,
        "y_train": y_train,
        "y_test": y_test,
        "pred_train": pred_train,
        "pred_test": pred_test,
        "std_test": std_test,
        "metrics": metrics,
        "predictions": predictions,
    }


def parity_plot(result, target_label, filename):
    y_train = result["y_train"].to_numpy()
    y_test = result["y_test"].to_numpy()
    pred_train = result["pred_train"]
    pred_test = result["pred_test"]
    std_test = result["std_test"]

    all_values = np.concatenate([y_train, y_test, pred_train, pred_test])
    lower = min(0.0, float(all_values.min()))
    upper = float(all_values.max()) * 1.06

    fig, ax = plt.subplots(figsize=(6.2, 5.5), dpi=160)
    ax.scatter(y_train, pred_train, s=45, color="purple", alpha=0.45, edgecolor="black", linewidth=0.4, label="Train")
    ax.errorbar(y_test, pred_test, yerr=1.96 * std_test, fmt="o", ms=6, color="teal", ecolor="teal", alpha=0.75, capsize=2, label="Test ±1.96 SD")
    ax.plot([lower, upper], [lower, upper], "--", color="black", linewidth=1.2, label="1:1")
    ax.set(xlabel=f"Measured {target_label}", ylabel=f"Predicted {target_label}", xlim=(lower, upper), ylim=(lower, upper))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 4. GP model for BHET yield

In [ ]:
yield_result = fit_and_evaluate("yield", TARGETS["yield"])
display(pd.DataFrame([yield_result["metrics"]]))
parity_plot(yield_result, "BHET yield (mg BHET g PET$^{-1}$)", "yield_parity.png")

## 5. GP model for BHET selectivity

In [ ]:
# Mirrored workflow: identical features, split, kernel, alpha, normalization,
# optimizer restarts, and random states; only the response column changes.
selectivity_result = fit_and_evaluate("selectivity", TARGETS["selectivity"])
display(pd.DataFrame([selectivity_result["metrics"]]))
parity_plot(selectivity_result, "BHET selectivity (%)", "selectivity_parity.png")

## 6. Combined model metrics and test predictions

In [ ]:
metrics_df = pd.DataFrame([
    yield_result["metrics"],
    selectivity_result["metrics"],
])
predictions_df = pd.concat([
    yield_result["predictions"],
    selectivity_result["predictions"],
], ignore_index=True)

metrics_df.to_csv(OUTPUT_DIR / "gp_metrics.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "gp_test_predictions.csv", index=False)

display(metrics_df)
display(predictions_df.head())

## Part II — SHAP interpretation of internally trained GP models

SHAP is calculated separately for each GP fitted to the internal training partition. This model-agnostic `KernelExplainer` uses 30 background points and 100 sampled evaluations in the original code. Interpretations correspond to the internally trained model, not the full-data refit.

In [ ]:
def compute_shap(result, target_name):
    model = result["model"]
    background = shap.sample(X_train, 30, random_state=SHAP_RANDOM_STATE)
    explainer = shap.KernelExplainer(model.predict, background)
    values = np.asarray(explainer.shap_values(X, nsamples=100, silent=True))
    if values.ndim == 3:
        values = values[..., 0]
    explanation = shap.Explanation(
        values=values,
        base_values=np.repeat(float(np.asarray(explainer.expected_value).reshape(-1)[0]), len(X)),
        data=X.to_numpy(),
        feature_names=FEATURES,
    )
    importance = pd.DataFrame({
        "target": target_name,
        "feature": FEATURES,
        "mean_abs_shap": np.abs(values).mean(axis=0),
        "mean_shap": values.mean(axis=0),
    }).sort_values("mean_abs_shap", ascending=False, ignore_index=True)
    return explanation, importance


yield_shap, yield_shap_importance = compute_shap(yield_result, "yield")
selectivity_shap, selectivity_shap_importance = compute_shap(selectivity_result, "selectivity")

shap_importance_df = pd.concat([yield_shap_importance, selectivity_shap_importance], ignore_index=True)
shap_importance_df.to_csv(OUTPUT_DIR / "shap_feature_importance.csv", index=False)
display(shap_importance_df)

## 8. Global SHAP importance

In [ ]:
def shap_bar(explanation, target_label, filename):
    plt.close("all")
    plt.figure(figsize=(7, 4.8), dpi=160)
    shap.plots.bar(explanation, max_display=len(FEATURES), show=False)
    plt.title(f"Global SHAP importance: {target_label}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close("all")


shap_bar(yield_shap, "BHET yield", "yield_shap_bar.png")
shap_bar(selectivity_shap, "BHET selectivity", "selectivity_shap_bar.png")

## 9. SHAP effect direction and heterogeneity

In [ ]:
def shap_beeswarm(explanation, target_label, filename):
    plt.close("all")
    plt.figure(figsize=(7.5, 5.0), dpi=160)
    shap.plots.beeswarm(explanation, max_display=len(FEATURES), show=False)
    plt.title(f"SHAP effects: {target_label}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close("all")


shap_beeswarm(yield_shap, "BHET yield", "yield_shap_beeswarm.png")
shap_beeswarm(selectivity_shap, "BHET selectivity", "selectivity_shap_beeswarm.png")

## 10. Dependence of model contribution on ethylene glycol

In [ ]:
def dependence_plot(explanation, target_label, filename):
    plt.close("all")
    plt.figure(figsize=(7.0, 5.0), dpi=160)
    shap.plots.scatter(explanation[:, "solvent"], color=explanation, show=False)
    plt.xlabel("Ethylene glycol (vol%)")
    plt.ylabel(f"SHAP contribution to {target_label}")
    plt.title(f"EG dependence: {target_label}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close("all")


dependence_plot(yield_shap, "BHET yield", "yield_shap_EG_dependence.png")
dependence_plot(selectivity_shap, "BHET selectivity", "selectivity_shap_EG_dependence.png")

## 11. Local SHAP explanations

In [ ]:
yield_example_index = int(data[TARGETS["yield"]].idxmax())
selectivity_example_index = int(data[TARGETS["selectivity"]].idxmax())

print("Highest-yield seed row:", yield_example_index)
display(data.loc[[yield_example_index], FEATURES + [TARGETS["yield"], TARGETS["selectivity"]]])
plt.close("all")
plt.figure(figsize=(8, 4.8), dpi=160)
shap.plots.waterfall(yield_shap[yield_example_index], max_display=len(FEATURES), show=False)
plt.title("Local explanation: highest-yield seed condition")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "yield_shap_local_waterfall.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close("all")

print("Highest-selectivity seed row:", selectivity_example_index)
display(data.loc[[selectivity_example_index], FEATURES + [TARGETS["yield"], TARGETS["selectivity"]]])
plt.close("all")
plt.figure(figsize=(8, 4.8), dpi=160)
shap.plots.waterfall(selectivity_shap[selectivity_example_index], max_display=len(FEATURES), show=False)
plt.title("Local explanation: highest-selectivity seed condition")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "selectivity_shap_local_waterfall.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close("all")

## 12. Programmatic interpretation summary

In [ ]:
def interpretation_summary(importance, explanation, target_name):
    top = importance.iloc[0]
    eg_idx = FEATURES.index("solvent")
    eg_values = X["solvent"].to_numpy()
    eg_shap = explanation.values[:, eg_idx]
    low = eg_shap[eg_values <= 20]
    high = eg_shap[eg_values >= 40]
    return {
        "target": target_name,
        "top_feature": top["feature"],
        "top_mean_abs_shap": float(top["mean_abs_shap"]),
        "mean_EG_SHAP_at_0_to_20_vol_pct": float(low.mean()) if len(low) else np.nan,
        "mean_EG_SHAP_at_40_to_50_vol_pct": float(high.mean()) if len(high) else np.nan,
        "interpretive_note": "SHAP describes the fitted model, not a causal mechanism.",
    }


interpretation_df = pd.DataFrame([
    interpretation_summary(yield_shap_importance, yield_shap, "yield"),
    interpretation_summary(selectivity_shap_importance, selectivity_shap, "selectivity"),
])
interpretation_df.to_csv(OUTPUT_DIR / "shap_interpretation_summary.csv", index=False)
display(interpretation_df)

## Interpretation guardrails

- SHAP values explain the GP predictions within the constrained 203-point seed dataset.
- A large SHAP magnitude means that a variable strongly changes the model output relative to the background prediction; it does not establish causality.
- Correlated or unevenly sampled variables can share or distort attribution.
- Interpretation of high-performing hold-out/BO-selected candidates requires applying the fitted models to those candidates separately and assessing predictive uncertainty and domain shift.

## Part III — External hold-out validation and uncertainty calibration

**Protocol:** harmonize the input schemas, check equivalent substrate-loading encodings, refit **the unchanged GP specification** on the full constrained dataset (`n=203`), and use the independent hold-out dataset (`n=237`) only for evaluation. The target definitions and units remain those of the original workbooks.

The code below retains the original uncertainty-calibration metrics and diagnostic plots. In particular, nominal predictive intervals derived from the GP standard deviation describe **model uncertainty under the specified GP assumptions**; they should not be conflated with guaranteed experimental predictive coverage.

### 13. Configure hold-out dataset and outputs

Output artifacts for the hold-out analysis are kept in a separate subfolder to avoid overwriting internal-validation files.

In [ ]:
SEED_PATH = DATA_PATH
HOLDOUT_PATH = resolve_input("holdout_237.xlsx")
OUTPUT_DIR = ROOT / "gp_calibration_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Constrained dataset:", SEED_PATH.resolve())
print("Hold-out dataset:", HOLDOUT_PATH.resolve())
print("Calibration outputs:", OUTPUT_DIR.resolve())

## 1. Load and harmonize the two schemas

In [ ]:
seed_raw = pd.read_excel(SEED_PATH, sheet_name="constrained")
holdout_raw = pd.read_excel(HOLDOUT_PATH, sheet_name="clean")

seed = seed_raw.rename(columns={
    "S": "S_mass",
    "time": "time",
    "enzyme": "enzyme",
    "solvent": "EG",
    "temperature": "temperature",
    "ph": "pH",
    "substrate / disc mL-1": "substrate_discs",
    "Mean_BHET": "yield",
    "Mean_Selectivity_BHET": "selectivity",
}).copy()

holdout = holdout_raw.rename(columns={
    "S": "S_mass",
    "time": "time",
    "E": "enzyme",
    "EG": "EG",
    "T": "temperature",
    "ph": "pH",
    "substrate / disc mL-1": "substrate_discs",
    "BHET Yield": "yield",
    "BHET Selectivity": "selectivity",
}).copy()

FEATURES = ["S_mass", "time", "enzyme", "EG", "temperature", "pH"]
TARGETS = ["yield", "selectivity"]
required = FEATURES + ["substrate_discs"] + TARGETS

for name, frame in {"seed": seed, "holdout": holdout}.items():
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise KeyError(f"{name} lacks required columns: {missing}")
    if frame[required].isna().any().any():
        raise ValueError(f"{name} has missing modelling values")
    if frame.duplicated(FEATURES).any():
        raise ValueError(f"{name} has duplicate six-variable conditions")

print("Seed shape:", seed.shape)
print("Hold-out shape:", holdout.shape)
display(seed[required].head())
display(holdout[required].head())

## 2. Resolve the substrate encoding

In [ ]:
substrate_check = pd.concat([
    seed[["S_mass", "substrate_discs"]].assign(dataset="seed"),
    holdout[["S_mass", "substrate_discs"]].assign(dataset="holdout"),
], ignore_index=True)
substrate_check["mass_per_disc"] = substrate_check["S_mass"] / substrate_check["substrate_discs"]

substrate_summary = substrate_check.groupby("dataset").agg(
    rows=("S_mass", "size"),
    unique_mass_levels=("S_mass", "nunique"),
    unique_disc_levels=("substrate_discs", "nunique"),
    minimum_mass_per_disc=("mass_per_disc", "min"),
    maximum_mass_per_disc=("mass_per_disc", "max"),
).reset_index()

exact_linear_encoding = bool(np.allclose(substrate_check["S_mass"], 32.0 * substrate_check["substrate_discs"], rtol=0, atol=1e-12))
print("S_mass == 32 * substrate_discs for every row:", exact_linear_encoding)
display(substrate_summary)
substrate_summary.to_csv(OUTPUT_DIR / "substrate_encoding_check.csv", index=False)

if not exact_linear_encoding:
    raise ValueError("Substrate mass and disc loading are not exactly proportional.")

`S_mass` and disc loading contain identical information because `S_mass = 32 × substrate_discs` for every seed and hold-out row. The notebook retains `S_mass` to reproduce the original yield code exactly. Predictions are invariant to using disc loading after GP length-scale re-optimization, but the manuscript and released code should use one name and unit consistently.

## 3. Fit unchanged GPs to all 203 seed points

In [ ]:
# Reuse make_gp() defined in Part I; GP kernel and settings remain identical.
X_seed = seed[FEATURES]
X_holdout = holdout[FEATURES]
models = {}

for target in TARGETS:
    model = make_gp()
    model.fit(X_seed, seed[target])
    models[target] = model
    print(target, "fitted kernel:", model.kernel_)

## 4. Generate posterior predictions and residual diagnostics

In [ ]:
prediction_frames = []

for target, model in models.items():
    mean, sd = model.predict(X_holdout, return_std=True)
    observed = holdout[target].to_numpy()
    residual = observed - mean
    safe_sd = np.maximum(sd, np.finfo(float).eps)
    standardized = residual / safe_sd

    result = holdout[FEATURES + ["substrate_discs"]].copy()
    result.insert(0, "holdout_row", np.arange(len(holdout)))
    result.insert(1, "target", target)
    result["observed"] = observed
    result["posterior_mean"] = mean
    result["posterior_sd"] = sd
    result["lower_68"] = mean - sd
    result["upper_68"] = mean + sd
    result["lower_95"] = mean - 1.96 * sd
    result["upper_95"] = mean + 1.96 * sd
    result["residual"] = residual
    result["standardized_residual"] = standardized
    result["within_68"] = np.abs(standardized) <= 1.0
    result["within_95"] = np.abs(standardized) <= 1.96
    result["absolute_error"] = np.abs(residual)
    prediction_frames.append(result)

predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_csv(OUTPUT_DIR / "holdout_predictions.csv", index=False)
display(predictions.head())

## 5. Overall predictive accuracy and calibration

In [ ]:
def calibration_metrics(group):
    observed = group["observed"].to_numpy()
    mean = group["posterior_mean"].to_numpy()
    sd = np.maximum(group["posterior_sd"].to_numpy(), np.finfo(float).eps)
    z = group["standardized_residual"].to_numpy()
    nlpd = 0.5 * np.log(2 * np.pi * sd**2) + 0.5 * z**2
    return pd.Series({
        "n": len(group),
        "r2": r2_score(observed, mean),
        "rmse": mean_squared_error(observed, mean) ** 0.5,
        "mae": np.mean(np.abs(observed - mean)),
        "coverage_68": group["within_68"].mean(),
        "coverage_95": group["within_95"].mean(),
        "expected_coverage_68": 0.6827,
        "expected_coverage_95": 0.95,
        "mean_posterior_sd": sd.mean(),
        "mean_95_interval_width": np.mean(3.92 * sd),
        "mean_residual": np.mean(observed - mean),
        "mean_standardized_residual": z.mean(),
        "sd_standardized_residual": z.std(ddof=1),
        "median_absolute_standardized_residual": np.median(np.abs(z)),
        "mean_nlpd": nlpd.mean(),
    })


calibration_summary = predictions.groupby("target", sort=False).apply(calibration_metrics, include_groups=False).reset_index()
calibration_summary.to_csv(OUTPUT_DIR / "calibration_summary.csv", index=False)
display(calibration_summary)

### Interpretation criteria

For an approximately calibrated Gaussian posterior, about 68.3% and 95% of observations should lie within ±1 SD and ±1.96 SD, respectively. A standardized-residual mean near zero indicates little average bias, while a standard deviation near one indicates appropriate uncertainty scale. Coverage substantially below nominal and standardized-residual spread above one indicate overconfident uncertainty estimates or domain shift.

## 6. Calibration by reaction variable

In [ ]:
grouping_variables = ["EG", "pH", "temperature", "time"]
group_rows = []

for target in TARGETS:
    target_data = predictions[predictions["target"] == target]
    for variable in grouping_variables:
        for level, group in target_data.groupby(variable):
            row = calibration_metrics(group).to_dict()
            row.update({"target": target, "variable": variable, "level": level})
            group_rows.append(row)

calibration_by_variable = pd.DataFrame(group_rows)
calibration_by_variable = calibration_by_variable[["target", "variable", "level"] + [
    column for column in calibration_by_variable.columns if column not in {"target", "variable", "level"}
]]
calibration_by_variable.to_csv(OUTPUT_DIR / "calibration_by_variable.csv", index=False)
display(calibration_by_variable[calibration_by_variable["variable"].isin(["EG", "pH", "temperature"])])

## 7. External parity plots with posterior intervals

In [ ]:
def external_parity(target, label, filename):
    frame = predictions[predictions["target"] == target]
    observed = frame["observed"].to_numpy()
    mean = frame["posterior_mean"].to_numpy()
    sd = frame["posterior_sd"].to_numpy()
    lower = min(0.0, float(min(observed.min(), mean.min())))
    upper = float(max(observed.max(), mean.max())) * 1.06

    fig, ax = plt.subplots(figsize=(6.2, 5.5), dpi=160)
    ax.errorbar(observed, mean, yerr=1.96 * sd, fmt="o", ms=4.5, alpha=0.55, color="#0F766E", ecolor="#5EEAD4", capsize=1.5)
    ax.plot([lower, upper], [lower, upper], "--", color="black", linewidth=1.2)
    ax.set(xlabel=f"Measured {label}", ylabel=f"Seed-GP posterior mean: {label}", xlim=(lower, upper), ylim=(lower, upper))
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)


external_parity("yield", "BHET yield (mg BHET g PET$^{-1}$)", "holdout_yield_parity.png")
external_parity("selectivity", "BHET selectivity (%)", "holdout_selectivity_parity.png")

## 8. Standardized-residual calibration plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=160)
for ax, target, color in zip(axes, TARGETS, ["#7C3AED", "#0F766E"]):
    z = predictions.loc[predictions["target"] == target, "standardized_residual"]
    sns.histplot(z, bins=30, stat="density", color=color, alpha=0.55, ax=ax)
    grid = np.linspace(-4, 4, 300)
    normal = np.exp(-0.5 * grid**2) / np.sqrt(2 * np.pi)
    ax.plot(grid, normal, "--", color="black", label="N(0,1)")
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline(-1.96, color="gray", linestyle=":")
    ax.axvline(1.96, color="gray", linestyle=":")
    ax.set(title=target.capitalize(), xlabel="Standardized residual", ylabel="Density", xlim=(-10, 10))
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("External hold-out standardized residuals")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "standardized_residual_histograms.png", bbox_inches="tight")
plt.show()
plt.close(fig)

## 9. Residual-versus-variable diagnostics

In [ ]:
def residual_grid(target, filename):
    frame = predictions[predictions["target"] == target]
    variables = ["EG", "pH", "temperature", "time", "posterior_mean", "posterior_sd"]
    labels = ["EG (vol%)", "pH", "Temperature (°C)", "Time (h)", "Posterior mean", "Posterior SD"]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8), dpi=150)
    for ax, variable, label in zip(axes.flat, variables, labels):
        ax.scatter(frame[variable], frame["standardized_residual"], s=24, alpha=0.55, color="#2563EB", edgecolor="none")
        ax.axhline(0, color="black", linewidth=1)
        ax.axhline(-1.96, color="gray", linestyle=":")
        ax.axhline(1.96, color="gray", linestyle=":")
        ax.set(xlabel=label, ylabel="Standardized residual")
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"External residual diagnostics: {target}")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.show()
    plt.close(fig)


residual_grid("yield", "yield_residual_diagnostics.png")
residual_grid("selectivity", "selectivity_residual_diagnostics.png")

## 10. Coverage and bias across EG concentration

In [ ]:
eg_summary = calibration_by_variable[calibration_by_variable["variable"] == "EG"].copy()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=150, sharex=True)
for column, target in enumerate(TARGETS):
    frame = eg_summary[eg_summary["target"] == target].sort_values("level")
    axes[0, column].plot(frame["level"], frame["coverage_95"], marker="o", color="#0F766E")
    axes[0, column].axhline(0.95, color="black", linestyle="--", label="Nominal 95%")
    axes[0, column].set(title=target.capitalize(), ylabel="95% coverage", ylim=(0, 1.05))
    axes[0, column].legend(frameon=False)
    axes[1, column].plot(frame["level"], frame["mean_residual"], marker="o", color="#7C3AED")
    axes[1, column].axhline(0, color="black", linestyle="--")
    axes[1, column].set(xlabel="EG (vol%)", ylabel="Mean residual (observed − mean)")
for ax in axes.flat:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("External calibration and bias by EG concentration")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "calibration_by_EG.png", bbox_inches="tight")
plt.show()
plt.close(fig)
display(eg_summary[["target", "level", "n", "coverage_68", "coverage_95", "mean_residual", "mean_standardized_residual"]])

## 11. Most severe extrapolation errors

In [ ]:
largest_errors = (
    predictions.assign(abs_z=lambda frame: frame["standardized_residual"].abs())
    .sort_values(["target", "abs_z"], ascending=[True, False])
    .groupby("target", as_index=False)
    .head(15)
)
largest_errors.to_csv(OUTPUT_DIR / "largest_standardized_residuals.csv", index=False)
display(largest_errors[[
    "target", "holdout_row", *FEATURES, "observed", "posterior_mean", "posterior_sd",
    "residual", "standardized_residual", "within_95"
]])

## 12. Machine-readable interpretation summary

In [ ]:
interpretation_rows = []
for _, row in calibration_summary.iterrows():
    undercoverage_68 = row["coverage_68"] < row["expected_coverage_68"]
    undercoverage_95 = row["coverage_95"] < row["expected_coverage_95"]
    interpretation_rows.append({
        "target": row["target"],
        "external_r2": row["r2"],
        "coverage_68": row["coverage_68"],
        "coverage_95": row["coverage_95"],
        "mean_residual": row["mean_residual"],
        "mean_standardized_residual": row["mean_standardized_residual"],
        "sd_standardized_residual": row["sd_standardized_residual"],
        "undercoverage_68": bool(undercoverage_68),
        "undercoverage_95": bool(undercoverage_95),
        "direction_of_average_bias": "underprediction" if row["mean_residual"] > 0 else "overprediction",
        "interpretation": "External predictions evaluate deliberate domain shift from the response-constrained seed set; they are not an independent random test split.",
    })

interpretation_summary = pd.DataFrame(interpretation_rows)
interpretation_summary.to_csv(OUTPUT_DIR / "calibration_interpretation_summary.csv", index=False)
display(interpretation_summary)

report_lines = [
    "# External GP calibration summary",
    "",
    "Models were fitted on all 203 response-constrained seed points and evaluated once on all 237 external hold-out points.",
    "",
]
for _, row in calibration_summary.iterrows():
    report_lines.extend([
        f"## {row['target'].capitalize()}",
        "",
        f"- R²: {row['r2']:.4f}",
        f"- RMSE: {row['rmse']:.4f}",
        f"- Coverage within ±1 SD: {100*row['coverage_68']:.1f}% (nominal 68.3%)",
        f"- Coverage within ±1.96 SD: {100*row['coverage_95']:.1f}% (nominal 95.0%)",
        f"- Mean residual (observed − posterior mean): {row['mean_residual']:.4f}",
        f"- Standardized-residual mean / SD: {row['mean_standardized_residual']:.3f} / {row['sd_standardized_residual']:.3f}",
        "",
    ])
(OUTPUT_DIR / "calibration_report.md").write_text("\n".join(report_lines), encoding="utf-8")
print("\n".join(report_lines))

## Interpretation guardrails

- The hold-out set was defined by response thresholds and deliberately contains the higher-performing region excluded from seed initialization. It is therefore an external domain-shift benchmark, not an exchangeable random test set.
- Systematic positive residuals indicate underprediction; this is not inherently favorable. The relevant question is whether observations are compatible with posterior uncertainty.
- `alpha=0.001` is retained from the submitted model. If coverage is poor, noise modelling or post-hoc calibration should be evaluated transparently rather than silently tuning on the hold-out set.
- BO-selected-candidate calibration should be added after the exact five-cycle candidate list is reconstructed.